# A5 – Recomendaciones cuantificadas

El enunciado pide cuantificar al menos tres recomendaciones con la forma: *"Si hacemos X, esperamos un incremento/reducción de Y% en Z métrica, equivalente a [monto] en ingresos"*.

Cada recomendación sale de un hallazgo de A2, A3 o A4. Para cada una damos:
- el hallazgo en el que se apoya,
- el supuesto del que depende la cifra (y por qué es razonable),
- un escenario central y un rango (escenarios prudente y optimista),
- el riesgo principal.

Las cifras son **anuales**, tomando como base 2024 (el último año completo). Están ordenadas por el margen que recuperan. Las decisiones están en `DECISIONES.md` (A5.1 a A5.4).

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from src.carga import cargar_limpio, RAIZ

d = cargar_limpio()
cli = d["clientes"]
trx = d["transacciones"]
cam = d["campanas_marketing"]
anio = trx[trx.fecha >= "2024-01-01"]
VENTAS, MARGEN = anio.monto_neto.sum(), anio.margen_bruto.sum()

def miles(x):
    # 1.234 en vez de 1,234
    return f"{x:,.0f}".replace(",", ".")

def signo(x):
    return ("+" if x >= 0 else "−") + miles(abs(x))

print(f"Base 2024: ventas {miles(VENTAS)} | margen bruto {miles(MARGEN)} ({MARGEN / VENTAS:.1%})")
recomendaciones = []

Base 2024: ventas 8.684.581 | margen bruto 2.162.763 (24.9%)


## R1. Alinear el descuento online con el de tienda y app

**Hallazgo (A2.5):** los descuentos no generan volumen: con más descuento se compran las mismas unidades, también al comparar el mismo producto en distintos meses. Online da el doble de descuento que la tienda y la app (7,5% contra 3,7%) y es el canal con menor margen.

**Qué proponemos:** que el descuento medio online sea el mismo que en tienda y app. En la práctica: ninguna venta online con más descuento que el que tendría en tienda.

**Supuesto:** las unidades vendidas no cambian, porque el descuento no las mueve (A2.5). Entonces todo lo que se deja de regalar es ingreso y margen extra.

**Escenarios:** central = online iguala a tienda y app; prudente = se cierra la mitad de la brecha; optimista = no aplica (no proponemos bajar más). Además calculamos cuántas ventas online se podrían perder antes de que la medida deje de compensar.

In [2]:
online = anio[anio.canal == "online"]
resto = anio[anio.canal != "online"]
d_resto = resto.descuento.mean()
bruto = online.cantidad * online.precio_unitario          # venta a precio de lista
print(f"Descuento medio 2024: online {online.descuento.mean():.1%} | tienda y app {d_resto:.1%}")
print(f"Online regala {miles(online.costo_descuento.sum())} en descuentos ({online.costo_descuento.sum() / anio.costo_descuento.sum():.0%} del total)")

def recorte(meta):
    # cada venta online con más descuento que 'meta' baja a 'meta'
    nuevo = np.minimum(online.descuento, meta)
    return ((online.descuento - nuevo) * bruto).sum()

# Buscamos el tope que deja el descuento medio online igual al de tienda y app
topes = np.linspace(0, 0.35, 701)
medias = [np.minimum(online.descuento, t).mean() for t in topes]
tope = topes[np.argmin(np.abs(np.array(medias) - d_resto))]
central = recorte(tope)
prudente = recorte(topes[np.argmin(np.abs(np.array(medias) - (online.descuento.mean() + d_resto) / 2))])
print(f"\nTope por venta que iguala el descuento medio: {tope:.1%} (afecta a {(online.descuento > tope).mean():.0%} de las ventas online)")
print(f"Ingreso extra al año: central {miles(central)} | prudente (mitad de la brecha) {miles(prudente)}")

# ¿Cuántas ventas online se podrían perder antes de que deje de compensar?
margen_online = online.margen_bruto.sum()
perdida_max = central / (margen_online + central)
print(f"Margen bruto online 2024: {miles(margen_online)} -> {miles(margen_online + central)} (+{central / margen_online:.1%})")
print(f"Compensa mientras online no pierda más del {perdida_max:.0%} de sus ventas")

recomendaciones.append(["R1", "Alinear el descuento online con el de tienda y app (tope de "
                        f"{100 * tope:.1f}% por venta)".replace(".", ","), "margen bruto online", central / margen_online, central, prudente, central, central])

Descuento medio 2024: online 7.5% | tienda y app 3.7%
Online regala 274.235 en descuentos (57% del total)

Tope por venta que iguala el descuento medio: 4.7% (afecta a 58% de las ventas online)
Ingreso extra al año: central 141.067 | prudente (mitad de la brecha) 72.232
Margen bruto online 2024: 740.822 -> 881.890 (+19.0%)
Compensa mientras online no pierda más del 16% de sus ventas


## R2. Reducir el exceso de devoluciones de Ropa y Electrónica

**Hallazgo (A3.1):** solo dos categorías devuelven más de lo normal: Ropa y Calzado (21,8%) y Electrónica (17,7%), contra 10,3% del resto. El exceso viene de motivos concretos: en Ropa, la talla incorrecta y los defectuosos; en Electrónica, "no cumple expectativas" y los defectuosos.

**Qué proponemos:** guía de tallas y tabla de medidas en Ropa; fichas técnicas más completas (especificaciones, fotos reales, comparador) en Electrónica; y revisar a los proveedores con más defectuosos en las dos.

**Supuesto:** estas medidas recortan una parte del exceso sobre la tasa del resto de categorías. No sabemos cuánto: por eso damos escenarios de 25%, 50% y 75% del exceso. Cada devolución evitada es una venta que no se reembolsa, y su margen (según A2.1) no se pierde.

In [3]:
cats = ["Ropa_y_Calzado", "Electrónica"]
tasa_resto = anio[~anio.categoria.isin(cats)].devuelta.mean()
exc_ingreso = exc_margen = 0
for c in cats:
    x = anio[anio.categoria == c]
    fraccion = 1 - tasa_resto / x.devuelta.mean()     # parte de sus devoluciones que está por encima del resto
    dv = x[x.devuelta]
    exc_ingreso += dv.monto_neto.sum() * fraccion
    exc_margen += (dv.margen_bruto - dv.margen_neto).sum() * fraccion
    print(f"{c}: tasa {x.devuelta.mean():.1%} | exceso sobre el resto ({tasa_resto:.1%}): {fraccion:.0%} de sus devoluciones")
print(f"\nExceso de devoluciones en 2024: {miles(exc_ingreso)} en ingresos reembolsados, {miles(exc_margen)} de margen perdido")

esc = {f"{int(100 * p)}%": (p * exc_ingreso, p * exc_margen) for p in [0.25, 0.5, 0.75]}
for k, (i, m) in esc.items():
    print(f"  recortar el {k} del exceso: +{miles(i)} en ingresos, +{miles(m)} de margen")
dev_total = anio[anio.devuelta].monto_neto.sum()
print(f"Escenario central (50%): las devoluciones totales bajan un {esc['50%'][0] / dev_total:.1%}")

recomendaciones.append(["R2", "Guía de tallas en Ropa, mejores fichas en Electrónica y control de proveedores",
                        "monto devuelto", -esc["50%"][0] / dev_total, esc["50%"][0], esc["25%"][0], esc["75%"][0], esc["50%"][1]])

Ropa_y_Calzado: tasa 21.4% | exceso sobre el resto (10.4%): 51% de sus devoluciones
Electrónica: tasa 17.6% | exceso sobre el resto (10.4%): 41% de sus devoluciones

Exceso de devoluciones en 2024: 370.164 en ingresos reembolsados, 161.647 de margen perdido
  recortar el 25% del exceso: +92.541 en ingresos, +40.412 de margen
  recortar el 50% del exceso: +185.082 en ingresos, +80.824 de margen
  recortar el 75% del exceso: +277.623 en ingresos, +121.236 de margen
Escenario central (50%): las devoluciones totales bajan un 14.3%


## R3. Probar una campaña de reactivación para los Esporádicos en riesgo, con grupo de control

**Hallazgo (A3.4):** el segmento con más valor en riesgo es el Esporádico: 1.324 clientes con más de 180 días sin comprar, que dejarían de gastar 169.772 en 6 meses. La mitad vuelve sola. Y en A4.3 vimos que no hay pruebas de que las campañas actuales generen compras.

**Qué proponemos:** una campaña de reactivación para estos clientes, pero **dejando un grupo de control** (por ejemplo, un 20% al azar que no la recibe). Así, por primera vez, se mide si una campaña genera ventas comparando con quien no la recibió.

**Supuesto:** no sabemos cuánto recupera una campaña, y los datos actuales no permiten estimarlo. Damos escenarios de 5%, 10% y 20% de los ingresos en riesgo. El valor seguro de esta recomendación es la medición: sin ella, la empresa no puede saber si el marketing sirve.

In [4]:
riesgo_6m = 169_772          # A3, sección 3: ingresos en riesgo de los Esporádicos en 6 meses (notebook 03)
anual = 2 * riesgo_6m
ventas_esp = anio.merge(cli[["cliente_id", "segmento"]], on="cliente_id").query("segmento == 'Esporádico'").monto_neto.sum()
margen_pct = MARGEN / VENTAS
esc = {f"{int(100 * p)}%": p * anual for p in [0.05, 0.10, 0.20]}
for k, v in esc.items():
    print(f"  recuperar el {k} de lo que está en riesgo: +{miles(v)} en ingresos al año (+{miles(v * margen_pct)} de margen)")
print(f"Escenario central (10%): +{esc['10%'] / ventas_esp:.1%} en las ventas del segmento Esporádico ({miles(ventas_esp)} en 2024)")

recomendaciones.append(["R3", "Campaña de reactivación a Esporádicos en riesgo, con 20% de grupo de control",
                        "ventas del segmento Esporádico", esc["10%"] / ventas_esp, esc["10%"], esc["5%"], esc["20%"], esc["10%"] * margen_pct])

  recuperar el 5% de lo que está en riesgo: +16.977 en ingresos al año (+4.228 de margen)
  recuperar el 10% de lo que está en riesgo: +33.954 en ingresos al año (+8.456 de margen)
  recuperar el 20% de lo que está en riesgo: +67.909 en ingresos al año (+16.912 de margen)
Escenario central (10%): +2.5% en las ventas del segmento Esporádico (1.348.902 en 2024)


## R4. Limitar el descuento de las campañas al 5%

**Hallazgo (A4.5):** aunque aceptemos la marca de conversión, las conversiones extra que trae el descuento no pagan el descuento que se regala a todas las conversiones. Solo el 5% se paga solo; de 10% en adelante, cada envío pierde margen. Y según las compras reales, el descuento no cambia nada.

**Qué proponemos:** que ninguna campaña ofrezca más de 5% de descuento.

**Supuesto:** usamos la lectura más favorable al descuento (la marca de conversión de A4). Con la lectura de las compras reales el ahorro sería mayor, porque todo el descuento de campaña sería costo sin retorno. La conversión registrada bajaría, y hay que avisarlo para que no se lea como un fracaso.

In [5]:
pares = cam[cam.conversion_respaldada][["campana_id", "cliente_id", "fecha_envio"]].merge(
    trx[["cliente_id", "fecha", "monto_neto", "margen_bruto"]], on="cliente_id")
pares = pares[(pares.fecha - pares.fecha_envio).dt.days.between(0, 30)].sort_values("fecha").drop_duplicates("campana_id")
ticket, mpct = pares.monto_neto.mean(), pares.margen_bruto.sum() / pares.monto_neto.sum()
conv = cam.groupby("descuento_ofrecido").convertido.mean()
neto = (conv - conv[0]) * ticket * mpct - conv * conv.index * ticket       # margen por envío frente a 0% de descuento
envios = cam[cam.fecha_envio.dt.year == 2024].descuento_ofrecido.value_counts()
actual = (neto * envios).sum()
con_tope = neto[0.05] * envios[envios.index >= 0.05].sum()
ahorro = con_tope - actual
print("Margen por envío frente a no dar descuento (según la marca):", neto.round(2).to_dict())
print(f"Envíos 2024 con más de 5%: {envios[envios.index > 0.05].sum()} de {envios.sum()}")
print(f"Margen de las campañas 2024: actual {miles(actual)} | con tope de 5%: {miles(con_tope)} | diferencia +{miles(ahorro)}")
regalado = (cam[(cam.fecha_envio.dt.year == 2024)].pipe(lambda c: (c.convertido * c.descuento_ofrecido).sum()) * ticket)
regalado_5 = (cam[(cam.fecha_envio.dt.year == 2024)].pipe(lambda c: (c.descuento_ofrecido.clip(upper=0.05) > 0).mul(conv[0.05]).sum()) * 0.05 * ticket)
print(f"Descuento regalado en campañas 2024: {miles(regalado)} -> {miles(regalado_5)} con el tope")
conv_hoy = cam[cam.fecha_envio.dt.year == 2024].convertido.mean()
print(f"Conversión registrada esperada: cae de {conv[conv.index > 0.05].mean():.1%} a {conv[0.05]:.1%} en los envíos que hoy tienen más de 5%")

# Ingresos registrados (según la marca): conversiones × ticket × (1 − descuento). Con menos descuento hay menos conversiones.
altos = envios[envios.index > 0.05]
ingresos_cambio = sum(n * (conv[0.05] * ticket * 0.95 - conv[k] * ticket * (1 - k)) for k, n in altos.items())
print(f"Ingresos registrados por campañas: {'+' if ingresos_cambio > 0 else '−'}{miles(abs(ingresos_cambio))} al año (menos conversiones)")

recomendaciones.append(["R4", "Tope de 5% al descuento de las campañas", "descuento regalado en campañas",
                        -(regalado - regalado_5) / regalado, ingresos_cambio, ingresos_cambio, ingresos_cambio, ahorro])

Margen por envío frente a no dar descuento (según la marca): {0.0: 0.0, 0.05: 0.25, 0.1: -0.08, 0.15: -1.07, 0.2: -2.88, 0.25: -4.9}
Envíos 2024 con más de 5%: 2914 de 4452
Margen de las campañas 2024: actual -6.266 | con tope de 5%: 931 | diferencia +7.197
Descuento regalado en campañas 2024: 16.415 -> 4.507 con el tope
Conversión registrada esperada: cae de 10.6% a 6.9% en los envíos que hoy tienen más de 5%
Ingresos registrados por campañas: −21.737 al año (menos conversiones)


## Resumen: las recomendaciones en el formato del enunciado

In [6]:
r = pd.DataFrame(recomendaciones, columns=["id", "qué hacemos", "métrica", "cambio en la métrica", "ingresos al año (central)",
                                           "desde", "hasta", "margen al año (central)"]).set_index("id")
r = r.sort_values("margen al año (central)", ascending=False)
for i, f in r.iterrows():
    verbo = "un incremento" if f["cambio en la métrica"] > 0 else "una reducción"
    print(f"{i}. Si hacemos «{f['qué hacemos']}», esperamos {verbo} de {abs(f['cambio en la métrica']):.1%} en {f['métrica']}, "
          f"equivalente a {signo(f['ingresos al año (central)'])} en ingresos al año" +
          (f" (rango {miles(f['desde'])} a {miles(f['hasta'])})" if f["desde"] != f["hasta"] else "") +
          f" y {signo(f['margen al año (central)'])} de margen.")
print(f"\nTotal central: {miles(r['ingresos al año (central)'].sum())} en ingresos y {miles(r['margen al año (central)'].sum())} de margen al año "
      f"({r['margen al año (central)'].sum() / MARGEN:.1%} del margen bruto de 2024)")

# Tabla para el dashboard de Power BI (ver docs/dashboard.md)
from src.powerbi import guardar
guardar(r.reset_index(), "resultado_a5_recomendaciones")

R1. Si hacemos «Alinear el descuento online con el de tienda y app (tope de 4,7% por venta)», esperamos un incremento de 19.0% en margen bruto online, equivalente a +141.067 en ingresos al año (rango 72.232 a 141.067) y +141.067 de margen.
R2. Si hacemos «Guía de tallas en Ropa, mejores fichas en Electrónica y control de proveedores», esperamos una reducción de 14.3% en monto devuelto, equivalente a +185.082 en ingresos al año (rango 92.541 a 277.623) y +80.824 de margen.
R3. Si hacemos «Campaña de reactivación a Esporádicos en riesgo, con 20% de grupo de control», esperamos un incremento de 2.5% en ventas del segmento Esporádico, equivalente a +33.954 en ingresos al año (rango 16.977 a 67.909) y +8.456 de margen.
R4. Si hacemos «Tope de 5% al descuento de las campañas», esperamos una reducción de 72.5% en descuento regalado en campañas, equivalente a −21.737 en ingresos al año y +7.197 de margen.

Total central: 338.367 en ingresos y 237.544 de margen al año (11.0% del margen bruto de

## Notas

- **Orden de prioridad (por margen):** R1 (descuento online), R2 (devoluciones), R3 (reactivación con control) y R4 (descuento de campañas). Juntas suman 237.544 de margen al año en el escenario central, el 11,0% del margen bruto de 2024.
- **R1 es la más segura:** no requiere inversión y se apoya en un hallazgo comprobado de varias formas (A2.5). Su riesgo es que el descuento sí mueva ventas online y A2 no lo haya detectado: compensa mientras online no pierda más del 16% de sus ventas. Conviene aplicarla primero en una parte de los productos y medir.
- **R2 depende de cuánto funcionen las medidas.** Por eso damos un rango (25% a 75% del exceso). Tiene costos (guía de tallas, fichas, gestión de proveedores) que no conocemos y no restamos.
- **R3 es sobre todo una forma de medir.** Su valor seguro es que, con grupo de control, la empresa sabrá por fin si sus campañas generan ventas (A4.3). Las cifras son escenarios, no una estimación.
- **R4 es pequeña en dinero,** pero evita regalar descuento sin retorno. Hace caer la conversión registrada (de 10,6% a 6,9% en esos envíos), lo que hay que avisar para que no se lea como un fracaso.
- **No sumamos solapamientos importantes:** R1 actúa sobre el precio de las ventas online, R2 sobre las devoluciones de dos categorías, R3 sobre clientes que hoy no compran y R4 sobre el descuento de campaña. La única interacción es que R1 y R2 tocan algunas de las mismas ventas (Electrónica online); su efecto conjunto es algo menor que la suma.